In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import matplotlib.pyplot as plt

## RATING MAP
- A dictionary used to convert word ratings like "three" to an integer. Thus making the rating numerical for analysis

In [48]:
# Map rating words to numbers (for standardization)
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}


In [37]:
# Data collection from the 50 pages
base_url= "https://books.toscrape.com/catalogue/page-{}.html"

# container for scrapped books
records = []

In [38]:
# Scrape pages 1 through 50
for page in range(1, 51):                      
    url = base_url.format(page)                
    print(f"Fetching page {page}: {url}")     
    resp = requests.get(url, timeout=10)    
    # If request failed, raise an HTTPError so we notice
    resp.raise_for_status()

    # Parse HTML
    soup = BeautifulSoup(resp.text, "html.parser")

    # Find all book blocks (each book sits in article.product_pod)
    book_elems = soup.select("article.product_pod")
    print(f"  Found {len(book_elems)} book elements on page {page}")

    # Extract data from each book element
    for b in book_elems:
        # Title: stored in <h3><a title="...">
        title_tag = b.select_one("h3 a")
        title = title_tag.get("title").strip() if title_tag and title_tag.has_attr("title") else None

        # Price: in <p class="price_color">£51.77</p>
        price_tag = b.select_one("p.price_color")
        price_raw = price_tag.get_text(strip=True) if price_tag else None

        # Availability: <p class="instock availability"> In stock (20 available) </p>
        avail_tag = b.select_one("p.instock.availability")
        availability_raw = avail_tag.get_text(separator=" ", strip=True) if avail_tag else None

        # Rating: class on a <p class="star-rating Three"> -> extract the word 'Three'
        rating_tag = b.select_one("p.star-rating")
        rating_raw = None
        if rating_tag:
            classes = rating_tag.get("class", [])
            # classes usually like ['star-rating', 'Three'] -> pick the one that's not 'star-rating'
            rating_words = [c for c in classes if c.lower() != "star-rating"]
            rating_raw = rating_words[0] if rating_words else None

        # Append the raw extracted fields to records
        records.append({
            "title": title,
            "price_raw": price_raw,
            "availability_raw": availability_raw,
            "rating_raw": rating_raw
        })

    # polite pause to avoid hammering the server
    time.sleep(0.4)

# After scraping, convert to DataFrame
df = pd.DataFrame(records)
print("\nScraping finished. DataFrame shape:", df.shape)
print(df.head(5))


Fetching page 1: https://books.toscrape.com/catalogue/page-1.html
  Found 20 book elements on page 1
Fetching page 2: https://books.toscrape.com/catalogue/page-2.html
  Found 20 book elements on page 2
Fetching page 3: https://books.toscrape.com/catalogue/page-3.html
  Found 20 book elements on page 3
Fetching page 4: https://books.toscrape.com/catalogue/page-4.html
  Found 20 book elements on page 4
Fetching page 5: https://books.toscrape.com/catalogue/page-5.html
  Found 20 book elements on page 5
Fetching page 6: https://books.toscrape.com/catalogue/page-6.html
  Found 20 book elements on page 6
Fetching page 7: https://books.toscrape.com/catalogue/page-7.html
  Found 20 book elements on page 7
Fetching page 8: https://books.toscrape.com/catalogue/page-8.html
  Found 20 book elements on page 8
Fetching page 9: https://books.toscrape.com/catalogue/page-9.html
  Found 20 book elements on page 9
Fetching page 10: https://books.toscrape.com/catalogue/page-10.html
  Found 20 book element

In [40]:
# Show current columns and some info

print("Columns before cleaning:", df.columns.tolist())
print("Sample rows (raw):")
display(df.head(5))   


Columns before cleaning: ['title', 'price_raw', 'availability_raw', 'rating_raw']
Sample rows (raw):


,title,price_raw,availability_raw,rating_raw
0,A Light in the Attic,Â£51.77,In stock,Three
1,Tipping the Velvet,Â£53.74,In stock,One
2,Soumission,Â£50.10,In stock,One
3,Sharp Objects,Â£47.82,In stock,Four
4,Sapiens: A Brief History of Humankind,Â£54.23,In stock,Five


# Data Cleaning

In [43]:
# Cleaning price
df["price"] = (
    df["price_raw"]
    .astype(str)                              # ensure string operations work
    .str.replace("£", "", regex=False)        # remove pound symbol
    .str.replace(",", "", regex=False)        # remove any thousands commas (if present)
    .str.strip()                              # remove leading/trailing whitespace
)
df["price"] = pd.to_numeric(df["price"], errors="coerce")  # convert to numeric, set errors to NaN

In [44]:
# Cleaning availability
df["availability"] = df["availability_raw"].astype(str).str.strip()

In [49]:
#Standardize rating: map words to numbers using RATING_MAP
df["rating"] = df["rating_raw"].map(RATING_MAP)

In [50]:
# Handle missing values: check how many NaNs we have
print("\nMissing values per column after initial cleaning:")
print(df[["title", "price", "availability", "rating"]].isna().sum())



Missing values per column after initial cleaning:
title              0
price           1000
availability       0
rating             0
dtype: int64


In [51]:
# dropping rows without price 
df_clean = df.dropna(subset=["price"]).reset_index(drop=True)
print("\nAfter dropping rows with missing price:", df_clean.shape)



After dropping rows with missing price: (0, 7)


In [52]:
# Reorder and keep only useful columns for final dataset
final_cols = ["title", "price", "availability", "rating"]
df_final = df_clean[final_cols].copy()